# 5. Scoring predictions from any model

Predictions from any detector (not only ibbi's) can be scored with the benchmark's reference evaluator, as long as they are COCO results in the frame of the benchmark's COCO files.

In [ ]:
# Install PyTorch for your hardware first (https://pytorch.org), then:
%pip install -q ibbi

In [ ]:
import ibbi
from ibbi.evaluate.benchmark import evaluate_class_agnostic, evaluate_predictions, to_coco_results
root = ibbi.download_benchmark("iid_test")
test = ibbi.get_dataset("iid_test")

Make predictions with your own model (here an ibbi detector stands in). Species-level predictions need the benchmark's global category ids.

In [ ]:
import json
cats = {c["name"]: c["id"] for c in json.load(open(root / "detection/annotations_coco/train.json"))["categories"]}
det = ibbi.create_model("species_detector")
preds = []
for i in range(50):
    item = test[i]
    r = det.predict(item["image"], conf=0.001)
    preds += to_coco_results(item["image_id"], r["boxes"], r["scores"], [cats[s] for s in r["labels"]])

In [ ]:
res = evaluate_predictions({"iid_test": preds}, root)
res["headline"]

In [ ]:
evaluate_class_agnostic({"iid_test": preds}, root, operating_conf=0.25)["iid_test"]